## RAG for Hyderabad Institute of Technology 

In [ ]:
!pip install pypdf
!pip install langchain
!pip install langchain-community
!pip install sentence-transformers


In [14]:
import sys
print(sys.executable)

C:\Users\agarw\anaconda3\python.exe


In [15]:
from langchain_community.document_loaders import PyPDFLoader  # Used for loading the pdf
from pathlib import Path 

C:\Users\agarw\AppData\Local\Temp\ipykernel_20048\3251071915.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader  # Used for loading the pdf


In [4]:
knowledge_base=Path(r"C:\Users\agarw\OneDrive\Desktop\RAG project\Knowledge base")

# this will create a path object that will point to  folder

In [2]:
pdf_files=knowledge_base.glob("*.pdf")

# this will store all the files ending with .pdf into the pdf_files variable 


NameError: name 'knowledge_base' is not defined

In [16]:
documents=[]

In [21]:
# now we use for loop to deal with pdf one one at a time and also create empty documents list so that later we can add all the pdfs in one place 
#  we convert pdf into the string coz the pyPDFloader expects string 

for pdf in pdf_files:
    loader = PyPDFLoader(str(pdf))
    loaded_pages = loader.load()
    documents.extend(loaded_pages)
    

# only remove the comments when a new pdf is added to the folder

    
    

In [22]:
len(documents)

146

In [23]:
documents

[Document(metadata={'producer': 'Skia/PDF m152 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Document 10: Sports & Extracurricular Guidelines', 'source': 'C:\\Users\\agarw\\OneDrive\\Desktop\\RAG project\\Knowledge base\\Document 10_ Sports & Extracurricular Guidelines.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='Document  10:  Sports  &  Extracurricular  Guidelines  \nInstitution:  Hyderabad  Institute  of  Technology  (HIT)  \nDocument  Version:  2026.1  \nEffective  Academic  Year:  2026-2027  \n1.  Sports  Infrastructure  and  Operating  Hours  \nHyderabad  Institute  of  Technology  (HIT)  believes  in  the  holistic  development  of  its  \nengineering\n \nstudents.\n \nThe\n \ncampus\n \nhouses\n \nthe\n \nMajor\n \nDhyan\n \nChand\n \nIndoor\n \nSports\n \nComplex,\n \na\n \nfloodlit\n \noutdoor\n \nathletics\n \ntrack,\n \nand\n \na\n \nstate-of-the-art\n \ngymnasium.\n \nThese\n \nfacilities\n \nare\n \nmaintained\n \nthrough

In [24]:
## To replace the \n coz its causing each word to go to new line and make our text look bad we use this
import re
for doc in documents:
    doc.page_content=re.sub(r"\s+"," ",doc.page_content).strip() 
    
# Here we have replaved the whitespaces with a single space only in the page content of the doc
# Normalize all whitespace (spaces, tabs, newlines)
# into a single space and remove leading/trailing spaces.

In [29]:
print(documents)

[Document(metadata={'producer': 'Skia/PDF m152 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Document 10: Sports & Extracurricular Guidelines', 'source': 'C:\\Users\\agarw\\OneDrive\\Desktop\\RAG project\\Knowledge base\\Document 10_ Sports & Extracurricular Guidelines.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='Document 10: Sports & Extracurricular Guidelines Institution: Hyderabad Institute of Technology (HIT) Document Version: 2026.1 Effective Academic Year: 2026-2027 1. Sports Infrastructure and Operating Hours Hyderabad Institute of Technology (HIT) believes in the holistic development of its engineering students. The campus houses the Major Dhyan Chand Indoor Sports Complex, a floodlit outdoor athletics track, and a state-of-the-art gymnasium. These facilities are maintained through the annual sports fee collected during admission and are exclusively for the use of enrolled students and active faculty members. To ensure equitabl

In [26]:
print(documents)
len(documents)

[Document(metadata={'producer': 'Skia/PDF m152 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Document 10: Sports & Extracurricular Guidelines', 'source': 'C:\\Users\\agarw\\OneDrive\\Desktop\\RAG project\\Knowledge base\\Document 10_ Sports & Extracurricular Guidelines.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='Document 10: Sports & Extracurricular Guidelines Institution: Hyderabad Institute of Technology (HIT) Document Version: 2026.1 Effective Academic Year: 2026-2027 1. Sports Infrastructure and Operating Hours Hyderabad Institute of Technology (HIT) believes in the holistic development of its engineering students. The campus houses the Major Dhyan Chand Indoor Sports Complex, a floodlit outdoor athletics track, and a state-of-the-art gymnasium. These facilities are maintained through the annual sports fee collected during admission and are exclusively for the use of enrolled students and active faculty members. To ensure equitabl

146

In [12]:
print(documents[0].page_content)

Document 10: Sports & Extracurricular Guidelines Institution: Hyderabad Institute of Technology (HIT) Document Version: 2026.1 Effective Academic Year: 2026-2027 1. Sports Infrastructure and Operating Hours Hyderabad Institute of Technology (HIT) believes in the holistic development of its engineering students. The campus houses the Major Dhyan Chand Indoor Sports Complex, a floodlit outdoor athletics track, and a state-of-the-art gymnasium. These facilities are maintained through the annual sports fee collected during admission and are exclusively for the use of enrolled students and active faculty members. To ensure equitable access and maintain academic priorities, the sports facilities operate on a strictly regulated time schedule. The outdoor grounds and indoor courts are closed during peak academic instruction hours (9:30 AM to 4:00 PM). ● Morning Session: 5:30 AM to 8:30 AM (Open to all students). ● Evening Session: 4:30 PM to 9:00 PM (Reserved primarily for varsity team practic

In [10]:
## Perform Chunking on the PDF 

from langchain_text_splitters import RecursiveCharacterTextSplitter # used for chunking 

In [30]:
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=1300,
    chunk_overlap=450,
    length_function=len,
    separators=["\n\n","\n"," ",""]  #  It is a class and we have made a object for this
)

In [31]:
chunks = text_splitter.split_documents(documents) # now the dcoument is splitted into smaller smaller chunks 

In [32]:
len(chunks)

242

In [17]:
print(chunks[0])
print("="*100)
print(chunks[1])
print("="*100)
print(chunks[2])
print("="*100)
# print(chunks[])

page_content='Document 10: Sports & Extracurricular Guidelines Institution: Hyderabad Institute of Technology (HIT) Document Version: 2026.1 Effective Academic Year: 2026-2027 1. Sports Infrastructure and Operating Hours Hyderabad Institute of Technology (HIT) believes in the holistic development of its engineering students. The campus houses the Major Dhyan Chand Indoor Sports Complex, a floodlit outdoor athletics track, and a state-of-the-art gymnasium. These facilities are maintained through the annual sports fee collected during admission and are exclusively for the use of enrolled students and active faculty members. To ensure equitable access and maintain academic priorities, the sports facilities operate on a strictly regulated time schedule. The outdoor grounds and indoor courts are closed during peak academic instruction hours (9:30 AM to 4:00 PM). ● Morning Session: 5:30 AM to 8:30 AM (Open to all students). ● Evening Session: 4:30 PM to 9:00 PM (Reserved primarily for varsit

In [18]:
print(chunks[0].page_content)


Document 10: Sports & Extracurricular Guidelines Institution: Hyderabad Institute of Technology (HIT) Document Version: 2026.1 Effective Academic Year: 2026-2027 1. Sports Infrastructure and Operating Hours Hyderabad Institute of Technology (HIT) believes in the holistic development of its engineering students. The campus houses the Major Dhyan Chand Indoor Sports Complex, a floodlit outdoor athletics track, and a state-of-the-art gymnasium. These facilities are maintained through the annual sports fee collected during admission and are exclusively for the use of enrolled students and active faculty members. To ensure equitable access and maintain academic priorities, the sports facilities operate on a strictly regulated time schedule. The outdoor grounds and indoor courts are closed during peak academic instruction hours (9:30 AM to 4:00 PM). ● Morning Session: 5:30 AM to 8:30 AM (Open to all students). ● Evening Session: 4:30 PM to 9:00 PM (Reserved primarily for varsity team practic

In [19]:
print(chunks[0].metadata)

{'producer': 'Skia/PDF m152 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Document 10: Sports & Extracurricular Guidelines', 'source': 'C:\\Users\\agarw\\OneDrive\\Desktop\\RAG project\\Knowledge base\\Document 10_ Sports & Extracurricular Guidelines.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}


In [19]:
## EMBEDDINGS STARTS HERE 

from langchain_community.embeddings import HuggingFaceEmbeddings

In [7]:
# pip install sentence-transformers

In [20]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

C:\Users\agarw\AppData\Local\Temp\ipykernel_20048\2127729888.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [33]:
embeddings=embedding_model.embed_documents([chunk.page_content for chunk in chunks])  # embedding done

# We cretaed this just so we can see that how embeddings work and how they look like 
# FAISS creates is own embeddings and store them 


In [26]:
type(embeddings)


list

In [27]:
len(embeddings[0])

384

In [ ]:
pip install chromadb

In [ ]:
pip install langchain-chroma

In [22]:
from langchain_community.vectorstores import Chroma

In [ ]:
# UN_Comment only when you want to add new pdf or want to create new database

vector_store=Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="University_RAG",
    persist_directory=r"C:\Users\agarw\OneDrive\Desktop\RAG project\chroma_db"
)
## Vector Database created.

In [23]:
# use it to get info from the existing database.
vector_store = Chroma(
    persist_directory=r"C:\Users\agarw\OneDrive\Desktop\RAG project\chroma_db",
    embedding_function=embedding_model,
    collection_name="University_RAG"
)

C:\Users\agarw\AppData\Local\Temp\ipykernel_20048\1226687674.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(


In [33]:
type(vector_store)

langchain_community.vectorstores.chroma.Chroma

In [24]:
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k":5,
        "fetch_k":15,
        "lambda_mult":1
    }
)

In [25]:
from langchain_groq import ChatGroq

In [2]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [26]:
from dotenv import load_dotenv
import os

load_dotenv(r"C:\Users\agarw\OneDrive\Desktop\RAG project\.env")



True

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
C:\Users\agarw


In [27]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

In [28]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

In [29]:
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=retriever,
    llm=llm
)

In [30]:
from langchain_core.prompts import ChatPromptTemplate

In [31]:
prompt = ChatPromptTemplate.from_template("""
You are an AI assistant for Hyderabad Institute of Technology.

Answer the user's question using ONLY the provided context.

The question may contain multiple parts.
Answer every part that can be answered from the context.

If the context contains multiple policies or multiple values for the same concept, 
explain which policy each value belongs to instead of assuming they refer to the same situation.




Only if none of the required information exists in the context, reply exactly:
"I don't know based on the provided documents."

Do not make up information.

Keep the answer concise and accurate.
                                          
Context:
{context}

Question:
{question}







Answer:
""")

In [41]:
question="what is minimum cgpa required " 


# question=""

In [42]:
response=multi_query_retriever.invoke(question)

In [43]:
print(response[0].page_content)
print(response[0].metadata)

If a student's CGPA hits 9.6 but they failed a minor laboratory course (resulting in an F-grade backlog), they are immediately disqualified from the entire merit scholarship framework for that year, regardless of their high theoretical average.
{'creator': 'PyPDF', 'total_pages': 7, 'page_label': '3', 'creationdate': '', 'title': 'Document 16: Merit & Need-Based Scholarship Criteria', 'producer': 'Skia/PDF m152 Google Docs Renderer', 'source': 'C:\\Users\\agarw\\OneDrive\\Desktop\\RAG project\\Knowledge base\\Document 16_ Merit & Need-Based Scholarship Criteria.pdf', 'page': 2}


In [44]:
context="\n=========\n".join(doc.page_content for doc in response)

In [45]:
print(context)

If a student's CGPA hits 9.6 but they failed a minor laboratory course (resulting in an F-grade backlog), they are immediately disqualified from the entire merit scholarship framework for that year, regardless of their high theoretical average.
first day of freshman year up to the most recently completed semester. It uses the exact same weighted methodology, but applies it across all courses taken over multiple semesters. CGPA = (Sum of (Credits × Grade Points) for all completed courses) ÷ (Total Credits earned across all completed courses) Students aiming to convert their final graduation CGPA into a standard percentage equivalent for job applications must multiply their final CGPA by ten, as mandated by the institutional academic council.
of the following conditions at the end of their seventh semester: 1. The CGPA Floor: The student must maintain a Cumulative Grade Point Average (CGPA) strictly greater than or equal to 7.0. 2. Absolute Zero Backlogs: The student must have cleared ev

In [46]:
final_prompt = prompt.invoke(
    {
        "context": context,
        "question": question
    }
)

In [47]:
print(final_prompt.messages[0].content)


You are an AI assistant for Hyderabad Institute of Technology.

Answer the user's question using ONLY the provided context.

The question may contain multiple parts.
Answer every part that can be answered from the context.

If the context contains multiple policies or multiple values for the same concept, 
explain which policy each value belongs to instead of assuming they refer to the same situation.




Only if none of the required information exists in the context, reply exactly:
"I don't know based on the provided documents."

Do not make up information.

Keep the answer concise and accurate.
                                          
Context:
If a student's CGPA hits 9.6 but they failed a minor laboratory course (resulting in an F-grade backlog), they are immediately disqualified from the entire merit scholarship framework for that year, regardless of their high theoretical average.
first day of freshman year up to the most recently completed semester. It uses the exact same weig

In [48]:
answer = llm.invoke(final_prompt)

In [49]:
print(answer.content)

**Minimum CGPA requirements mentioned in the provided documents**

| Policy / Situation | Minimum CGPA required | How it is applied |
|--------------------|-----------------------|-------------------|
| **Seventh‑semester academic eligibility (CGPA Floor)** | **7.0** (or higher) | The student must maintain a cumulative GPA ≥ 7.0 through the end of the seventh semester. |
| **Campus‑placement portal access (Placement eligibility)** | **6.5** (calculated from semesters 1‑6) | At the start of the seventh semester, a student needs a cumulative GPA ≥ 6.5 for the first six semesters to be allowed onto the placement portal. |
| **Minor‑degree specialization entry** | **7.0** (continuously through the first two years) | To enter a minor in the 5th semester, the student must have a cumulative GPA ≥ 7.0 at all times during the first two years. |
| **Merit‑scholarship framework** | No explicit lower CGPA threshold is given; however, **any** failure in a minor laboratory (resulting in an F‑grade b

## EVALUATION

In [ ]:
## this code is upload the cvs file containing the question id and question and expected answers

import pandas as pd
 df=pd.read_csv(r"C:\Users\agarw\OneDrive\Desktop\RAG project\evaluation2.csv")
 df.head(5)

In [ ]:
 noW I WANT to create a loop that sends all these 50 questions to my llm one by one and then save the generated answerr in a file

result=[]

for index, row in df.iterrows():
    Question = row["Question"]
    Expected_Answer = row["Expected_Answer"]
    response=multi_query_retriever.invoke(Question)
    context="\n\n".join(doc.page_content for doc in response)
    final_prompt =prompt.invoke({
        "context":context,
        "question":Question
    })
    generated_answer=llm.invoke(final_prompt)
    generated_answer=generated_answer.content
    result.append({
        "Question_ID": row["Question_ID"],
        "Question":Question,
        "Expected_Answer":Expected_Answer,
        "generated_answer":generated_answer,
        "Retrived_Contents":context
    })
    
    
    
 

In [63]:
# evaluation_df=pd.DataFrame(result)

In [64]:
# evaluation_df.head(51)

In [65]:
# evaluation_df.to_csv(
#     r"C:\Users\agarw\OneDrive\Desktop\RAG project\evaluation_answer.csv",
#     index=False
# )

# print("Evaluation dataset created successfully!")


^C
Note: you may need to restart the kernel to use updated packages.


In [3]:
df=pd.read_csv(r"C:\Users\agarw\OneDrive\Desktop\RAG project\evaluation_answer.csv")
df.head(5)

,Question_ID,Question,Expected_Answer,generated_answer,Retrived_Contents
0,Q_001,What is the minimum CGPA required for a 3rd-ye...,A minimum CGPA of 7.5.,The minimum CGPA required for a third‑year stu...,"2. The ""AlumBridge"" Strategic Mentorship Frame..."
1,Q_002,What is the penalty if a student is caught log...,Both the student who provided the ID card and ...,If a student is caught logging proxy biometric...,"academic content, but they will be permanently..."
2,Q_003,If a Category-A student withdraws admission af...,The student will receive exactly a 50% refund ...,The student will receive **50 % of the paid tu...,will be returned within 48 hours. 2. Cancellat...
3,Q_004,What color must the hardbound thesis cover be ...,Dark Green fabric with Silver embossed lettering.,The hardbound thesis cover for a Civil Enginee...,"6. Thesis Formatting, Color Codes, and Final S..."
4,Q_005,Can a final-year student who has accepted a Ti...,No. Final-year students who have already accep...,No. Final‑year students who have already accep...,student) receive a mandatory ten-point bonus d...


In [162]:
# Using BERT-Score for semantic Evaluation

import pandas as pd
from bert_score import score


In [163]:
df=pd.read_csv(r"C:\Users\agarw\OneDrive\Desktop\RAG project\evaluation_answer.csv")

Bringing unauthorized firecrackers (classified as explosive materials) is prohibited as a weapon.If a student is caught with them, the university treats it as a zero‑tolerance weapons offense: the student is detained, handed over to the Cyberabad Police, permanently expelled, denied a degree and has the admission fee fully forfeited.


In [168]:
references = df["Expected_Answer"].tolist()
candidates = df["generated_answer"].tolist()

In [169]:
P,R,F1=score(candidates,references,lang="en",verbose=True)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


calculating scores...
computing bert embedding.


  0%|          | 0/2 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 31.06 seconds, 1.61 sentences/sec


In [172]:
print(R.mean())
print(F1.mean())
print(P.mean())

tensor(0.9322)
tensor(0.9035)
tensor(0.8768)


In [176]:
references = df["Expected_Answer"].tolist()
candidates = df["Retrived_Contents"].tolist()

In [177]:
P,R,F1=score(candidates,references,lang="en",verbose=True)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


calculating scores...
computing bert embedding.


  0%|          | 0/2 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 110.41 seconds, 0.45 sentences/sec


In [178]:
print(R.mean())
print(F1.mean())
print(P.mean())

tensor(0.8850)
tensor(0.8196)
tensor(0.7633)
